# E-commerce BI Chatbot — QLoRA Fine-Tuning (ALL 9 Categories)## TinyLlama-1.1B on Amazon Reviews 2023 (Real Data)**Person 2 — Model Fine-Tuning Lead | Group 25fltp**This notebook fine-tunes **TinyLlama-1.1B-Chat-v1.0** using **QLoRA** on **9 Amazon categories**.### Categories Covered1. Electronics2. Clothing_Shoes_and_Jewelry3. Home_and_Kitchen4. Books5. Sports_and_Outdoors6. Beauty_and_Personal_Care7. Toys_and_Games8. Food_and_Beverages9. Pet_Supplies### Task Types- SWOT Analysis- Competitor Comparison- Market Trends- Product Category Analysis- Customer Sentiment- Pricing & Delivery- Review Intelligence**Output:** GGUF model for Ollama on AWS EC2### Architecture```HuggingFace Dataset → Instruction Formatting → QLoRA Fine-Tuning → GGUF Export → S3 → EC2 Ollama```### Runtime- GPU: T4 (free Colab) — 16 GB VRAM- Time: ~30-40 min (2 epochs, ~7,000 examples per category)

## BLOCK 1: Install Dependencies

In [ ]:
# Install Unsloth (2x faster, 50% less VRAM)!pip install --quiet unsloth datasets sentencepiece tqdm# Verifyimport unslothprint(f'Unsloth version: {unsloth.__version__}')# GPU checkimport torchif torch.cuda.is_available():    import subprocess    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)    print(f'GPU: {result.stdout.strip()}')else:    print('WARNING: No GPU. Enable T4 GPU: Runtime > Change runtime type > T4 GPU')

## BLOCK 2: Define Categories and Load Data

In [ ]:
import pandas as pdimport requestsimport jsonimport random# 9 Amazon categoriesCATEGORIES = [    'Electronics',    'Clothing_Shoes_and_Jewelry',    'Home_and_Kitchen',    'Books',    'Sports_and_Outdoors',    'Beauty_and_Personal_Care',    'Toys_and_Games',    'Food_and_Beverages',    'Pet_Supplies',]DATASET = 'McAuley-Lab/Amazon-Reviews-2023'MAX_PER_CATEGORY = 10000def load_category(category, max_samples):    """Load reviews for one category from HuggingFace."""    url = (        f'https://huggingface.co/datasets/{DATASET}/resolve/main/raw/review_categories/'        f'{category}.jsonl'    )    print(f'Downloading {category} from HuggingFace...')    resp = requests.get(url, stream=True, timeout=120)    resp.raise_for_status()    reviews = []    for i, line in enumerate(resp.iter_lines(decode_unicode=True)):        if i >= max_samples:            break        if not line.strip():            continue        try:            rec = json.loads(line)            reviews.append({                'rating': rec.get('rating', 0),                'title': rec.get('title', ''),                'text': rec.get('text', ''),                'asin': rec.get('asin', ''),                'parent_asin': rec.get('parent_asin', ''),                'user_id': rec.get('user_id', ''),                'timestamp': rec.get('timestamp', ''),                'verified_purchase': rec.get('verified_purchase', ''),                'helpful_vote': rec.get('helpful_vote', 0),                'category': category,            })        except Exception:            continue    return pd.DataFrame(reviews)# Load ALL categoriesall_dfs = []for cat in CATEGORIES:    df_cat = load_category(cat, MAX_PER_CATEGORY)    all_dfs.append(df_cat)    print(f'  {cat}: {len(df_cat):,} reviews loaded')df = pd.concat(all_dfs, ignore_index=True)print(f'\nTotal: {len(df):,} reviews across {len(CATEGORIES)} categories')

## BLOCK 3: Clean & Filter Reviews

In [ ]:
# Drop missing textdf = df.dropna(subset=['text'])df = df[df['text'].str.strip() != '']# Remove very short (<50) and very long (>1500) reviewsdf = df[df['text'].str.len() >= 50]df = df[df['text'].str.len() <= 1500]# Deduplicatedf = df.drop_duplicates(subset=['user_id', 'asin', 'timestamp'])# Clean ratingsdf['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(3)# Add sentimentdef get_sentiment(r):    if r >= 4: return 'positive'    if r <= 2: return 'negative'    return 'neutral'df['sentiment'] = df['rating'].apply(get_sentiment)print(f'After cleaning: {len(df):,} reviews')print()print('By category:')print(df['category'].value_counts())print()print('Sentiment:')print(df['sentiment'].value_counts())

## BLOCK 4: Generate Instruction-Tuning Examples (7 Task Types)

In [ ]:
import randomrandom.seed(42)SYSTEM_PROMPT = (    'You are an expert e-commerce business intelligence analyst. '    'Answer based on review data and market knowledge. '    'Use structured formats: SWOT tables, bullet lists, comparison grids.')TASK_TEMPLATES = [    {        'task': 'swot_analysis',        'instruction': 'Perform a SWOT analysis for the e-commerce platform based on customer reviews. Include strategic takeaways.',    },    {        'task': 'competitor_comparison',        'instruction': 'Compare Amazon, Walmart, and Alibaba based on customer reviews. Use dimensions: Pricing, Logistics, Product Assortment, Customer Experience.',    },    {        'task': 'market_trends',        'instruction': 'Identify top 3 e-commerce market trends from customer review themes. Include business implications.',    },    {        'task': 'product_category',        'instruction': 'Analyze the key characteristics, customer preferences, and market dynamics of this product category.',    },    {        'task': 'customer_sentiment',        'instruction': 'Analyze customer sentiment patterns in the reviews below. Identify key themes, satisfaction drivers, and pain points.',    },    {        'task': 'pricing_delivery',        'instruction': 'Analyze customer sentiment around pricing and delivery based on reviews. Identify key pain points and satisfaction drivers.',    },    {        'task': 'review_intelligence',        'instruction': 'Summarize key product strengths and weaknesses from customer reviews. List top strengths and weaknesses with evidence.',    },]def generate_output(task, rating, sentiment, category):    """Generate reference output based on task type and metadata."""    if task == 'swot_analysis':        if rating >= 4:            strength = 'Strong product quality and customer satisfaction'            weakness = 'Price sensitivity and competitive pricing pressure'        elif rating <= 2:            strength = 'Large product selection and marketplace breadth'            weakness = 'Quality inconsistency and reliability issues'        else:            strength = 'Competitive pricing and value for money'            weakness = 'Variable customer service response times'        return (            '## Strengths\n'            f'- {strength}\n'            '- Established delivery infrastructure\n\n'            '## Weaknesses\n'            f'- {weakness}\n'            '- Regional service quality variation\n\n'            '## Opportunities\n'            '- Growing e-commerce market penetration\n'            '- AI-powered personalization\n\n'            '## Threats\n'            '- Intense competition from Walmart and Alibaba\n'            '- Supply chain disruptions\n\n'            '## Strategic Takeaway\n'            'Invest in quality assurance and logistics optimization.'        )    elif task == 'competitor_comparison':        return (            '| Dimension | Amazon | Walmart | Alibaba |\n'            '|---|---|---|---|\n'            '| Pricing | Competitive | Good value | Lowest prices |\n'            '| Logistics | Prime fast shipping | Same-day options | Variable |\n'            '| Assortment | Massive selection | Broad range | Huge marketplace |\n'            '| Customer Experience | Highly rated | Good | Mixed |\n\n'            '## Summary\n'            'Amazon leads in logistics and CX. Walmart excels in omnichannel. Alibaba dominates in price-sensitive global markets.'        )    elif task == 'market_trends':        return (            '## Trend 1: Same-Day Delivery Standardization\n'            '**Description**: Consumers expect same-day delivery as baseline.\n'            '**Business Implication**: Invest in micro-fulfillment centers.\n\n'            '## Trend 2: AI-Powered Product Recommendations\n'            '**Description**: ML models drive 35%+ of conversions.\n'            '**Business Implication**: Deploy recommendation engines.\n\n'            '## Trend 3: Social Commerce Integration\n'            '**Description**: Shoppable content accounts for 25%+ of discovery.\n'            '**Business Implication**: Build shoppable video integration.'        )    elif task == 'product_category':        return (            f'## Category: {category}\n\n'            f'### Key Characteristics\n'            '- Diverse product range within the category\n'            '- Price-performance trade-offs\n'            '- Brand loyalty patterns\n\n'            '### Customer Preferences\n'            '- Quality over quantity\n'            '- Value-driven purchasing decisions\n'            '- Convenience and delivery speed priorities\n\n'            '### Market Dynamics\n'            '- Seasonal demand fluctuations\n'            '- Competitive pricing pressure\n'            '- Product innovation cycles'        )    elif task == 'customer_sentiment':        return (            f'## Sentiment Analysis ({sentiment.capitalize()})\n\n'            '### Key Themes\n'            '- Product quality and value for money\n'            '- Shipping speed and reliability\n'            '- Customer service responsiveness\n\n'            f'### Overall Sentiment: {sentiment.capitalize()}\n'            '- Positive: Product quality, competitive pricing\n'            '- Negative: Delivery delays, quality inconsistencies'        )    elif task == 'pricing_delivery':        return (            f'## Pricing Sentiment\n'            f'- Overall: {sentiment.capitalize()}\n'            '- Key Driver: Value for money\n'            '- Pain Point: Hidden fees\n\n'            f'## Delivery Sentiment\n'            f'- Overall: {sentiment.capitalize()}\n'            '- Satisfaction: Shipping speed, tracking accuracy\n'            '- Complaints: Delivery delays, packaging damage'        )    else:  # review_intelligence        if rating >= 4:            return (                '## Key Strengths\n'                '1. Product Quality - Frequently praised\n'                '   Evidence: Durability and reliability mentions\n\n'                '## Key Weaknesses\n'                '1. Packaging - Environmental concerns\n'                '   Evidence: Excessive packaging mentions'            )        elif rating <= 2:            return (                '## Key Strengths\n'                '1. Product Selection - Wide variety\n'                '   Evidence: Range of options appreciated\n\n'                '## Key Weaknesses\n'                '1. Quality Consistency - Mixed quality\n'                '   Evidence: Inconsistent items received'            )        else:            return (                '## Key Strengths\n'                '1. Value for Money - Reasonable quality\n'                '   Evidence: Neutral reviews cite adequate functionality\n\n'                '## Key Weaknesses\n'                '1. Expectations Gap - Marketing vs reality\n'                '   Evidence: Description-reality discrepancies'            )def generate_example(row, template):    text = str(row['text'])[:600].strip()    rating = int(row['rating']) if row['rating'] >= 1 else 3    sentiment = row['sentiment']    category = row['category']    instruction = template['instruction']    output = generate_output(template['task'], rating, sentiment, category)    return {        'system': SYSTEM_PROMPT,        'input': instruction,        'context': f'Category: {category}\n' + text,        'output': output,    }print('Generating instruction-tuning examples...')examples = []for idx, row in df.iterrows():    template = random.choice(TASK_TEMPLATES)    try:        ex = generate_example(row, template)        examples.append(ex)    except Exception:        continueprint(f'Generated {len(examples):,} instruction-tuning examples')print()print('Example preview:')for k, v in list(examples[0].items()):    print(f'  {k}: {str(v)[:80]}...')

## BLOCK 5: Save as JSONL

In [ ]:
import jsonTRAIN_PATH = 'train.jsonl'with open(TRAIN_PATH, 'w', encoding='utf-8') as f:    for ex in examples:        f.write(json.dumps(ex, ensure_ascii=False) + '\n')print(f'Saved {len(examples):,} examples to {TRAIN_PATH}')# Verifywith open(TRAIN_PATH, 'r') as f:    lines = f.readlines()print(f'Verified: {len(lines):,} lines')

## BLOCK 6: Load TinyLlama with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModelimport torchMODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'MAX_SEQ_LENGTH = 2048DTYPE = None  # Auto-detect (float16 on T4)LOAD_IN_4BIT = Trueprint('Loading TinyLlama-1.1B with Unsloth...')print(f'Model: {MODEL_NAME}')print(f'CUDA available: {torch.cuda.is_available()}')model, tokenizer = FastLanguageModel.from_pretrained(    model_name=MODEL_NAME,    max_seq_length=MAX_SEQ_LENGTH,    dtype=DTYPE,    load_in_4bit=LOAD_IN_4BIT,)print('Model loaded!')

## BLOCK 7: Attach LoRA Adapters

In [ ]:
LORA_R = 16LORA_ALPHA = 32LORA_DROPOUT = 0.05LORA_TARGET_MODULES = [    'q_proj', 'k_proj', 'v_proj', 'o_proj',    'gate_proj', 'up_proj', 'down_proj',]print(f'Attaching LoRA: r={LORA_R}, alpha={LORA_ALPHA}')model = FastLanguageModel.get_peft_model(    model,    r=LORA_R,    lora_alpha=LORA_ALPHA,    lora_dropout=LORA_DROPOUT,    target_modules=LORA_TARGET_MODULES,    use_gradient_checkpointing='unsloth',)model.print_trainable_parameters()

## BLOCK 8: Load Dataset for SFTTrainer

In [ ]:
from datasets import load_datasettrain_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')print(f'Dataset: {len(train_dataset):,} examples')

## BLOCK 9: Format Chat Template

In [ ]:
def formatting_prompts_func(examples):    texts = []    EOS_TOKEN = tokenizer.eos_token    n = len(examples['system'])    for i in range(n):        text = (            '<|system|>' + examples['system'][i] + '\n'            '<|user|>' + examples['input'][i] + '\n'            '<|context|> ' + examples['context'][i] + '\n\n'            '<|assistant|>' + examples['output'][i] + EOS_TOKEN        )        texts.append(text)    return {'text': texts}train_dataset = train_dataset.map(    formatting_prompts_func,    batched=True,    remove_columns=['system', 'input', 'context', 'output'],)print(f'Formatted: {len(train_dataset):,} examples')print()print('Sample (first 200 chars):')print(train_dataset[0]['text'][:200])

## BLOCK 10: Training Arguments (Optimized for T4 GPU)

In [ ]:
from transformers import TrainingArgumentsEPOCHS = 2LEARNING_RATE = 2e-4BATCH_SIZE = 4GRADIENT_ACCUMULATION = 4  # Effective batch = 16WARMUP_STEPS = 10WEIGHT_DECAY = 0.01LOGGING_STEPS = 10OUTPUT_DIR = './outputs'training_args = TrainingArguments(    output_dir=OUTPUT_DIR,    per_device_train_batch_size=BATCH_SIZE,    gradient_accumulation_steps=GRADIENT_ACCUMULATION,    learning_rate=LEARNING_RATE,    num_train_epochs=EPOCHS,    warmup_steps=WARMUP_STEPS,    weight_decay=WEIGHT_DECAY,    logging_steps=LOGGING_STEPS,    save_strategy='epoch',    fp16=True,    optim='adamw_torch_fused',    report_to='none',    seed=42,    max_grad_norm=1.0,)print('Training Arguments:')print(f'  Epochs: {EPOCHS}')print(f'  Batch size: {BATCH_SIZE} (effective {BATCH_SIZE * GRADIENT_ACCUMULATION})')print(f'  Learning rate: {LEARNING_RATE}')print(f'  Mixed precision: fp16')print(f'  Output: {OUTPUT_DIR}')

## BLOCK 11: Fine-Tune with SFTTrainer

In [ ]:
import timefrom trl import SFTTrainerprint(f'Training on {len(train_dataset):,} examples...')print(f'Estimated time: ~{(len(train_dataset) * EPOCHS * 0.3 / 60):.1f} minutes\n')trainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=train_dataset,    args=training_args,    max_seq_length=MAX_SEQ_LENGTH,)start_time = time.time()trainer.train()elapsed = time.time() - start_timeprint(f'\nTraining complete in {elapsed/60:.1f} minutes')

## BLOCK 12: Export to GGUF for Ollama

In [ ]:
import os# Unsloth 2026.4.8 creates QUADRUPLE-nested path: ./outputs/ecom_chatbot_gguf_gguf_gguf_gguf/GGUF_OUTPUT_DIR = './outputs/ecom_chatbot_gguf_gguf_gguf_gguf'os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)print('Exporting to GGUF (Q4_K_M quantization)...')print(f'Output: {GGUF_OUTPUT_DIR}')model.save_pretrained_gguf(    GGUF_OUTPUT_DIR,    tokenizer,    quantization_method='q4_k_m',)print('GGUF export complete!')# Find the GGUF filegguf_file = Nonefor root, dirs, files in os.walk('./outputs'):    for f in files:        if f.endswith('.gguf'):            gguf_file = f            gguf_full = os.path.join(root, f)            break    if gguf_file:        breakprint(f'GGUF file: {gguf_file}')print(f'Full path: {gguf_full}')import ossize_mb = os.path.getsize(gguf_full) / (1024 * 1024)print(f'Size: {size_mb:.1f} MB')

## BLOCK 13: Download GGUF File

In [ ]:
from google.colab import filesprint(f'Downloading GGUF from: {gguf_full}')files.download(gguf_full)print('Download started — check browser download bar')

## BLOCK 14: Upload GGUF to S3 (for EC2 Deployment)

In [ ]:
import subprocessS3_BUCKET = '25fltp-ecom-chatbot'S3_KEY = 'model/tinyllama-chat.Q4_K_M.gguf'print('Uploading GGUF to S3...')result = subprocess.run(    ['aws', 's3', 'cp', gguf_full, f's3://{S3_BUCKET}/{S3_KEY}'],    capture_output=True, text=True)print('STDOUT:', result.stdout)print('STDERR:', result.stderr)# Verifyverify = subprocess.run(    ['aws', 's3', 'ls', f's3://{S3_BUCKET}/{S3_KEY}'],    capture_output=True, text=True)print('S3 contents:')print(verify.stdout)

## BLOCK 15: Inference Test

In [ ]:
from unsloth import FastLanguageModelimport torch# Reload for inference (model is already fine-tuned in memory)inference_model = FastLanguageModel.for_inference(model)TEST_PROMPTS = [    {        'task': 'SWOT Analysis',        'text': (            '<|system|>You are an expert e-commerce business intelligence analyst.\n'            '<|user|>Give me a SWOT analysis for Amazon in e-commerce.\n'            '<|context|> Amazon is the largest e-commerce platform globally with $500B+ revenue.\n\n'            '<|assistant|>'        ),    },    {        'task': 'Competitor Comparison',        'text': (            '<|system|>You are an expert e-commerce business intelligence analyst.\n'            '<|user|>Compare Amazon, Walmart, and Alibaba on pricing, logistics, and AI.\n'            '<|context|> These three companies dominate global e-commerce with different strategies.\n\n'            '<|assistant|>'        ),    },    {        'task': 'Market Trends',        'text': (            '<|system|>You are an expert e-commerce business intelligence analyst.\n'            '<|user|>What are the current e-commerce market trends in 2025?\n'            '<|context|> E-commerce continues to grow with new technologies and consumer behaviors.\n\n'            '<|assistant|>'        ),    },]print('Testing fine-tuned model...\n')for prompt in TEST_PROMPTS:    print(f'=== {prompt["task"]} ===')    inputs = tokenizer([prompt['text']], return_tensors='pt').to('cuda')    outputs = inference_model.generate(        **inputs,        max_new_tokens=256,        temperature=0.3,        top_p=0.9,        use_cache=True,    )    result = tokenizer.decode(outputs[0], skip_special_tokens=False)    # Extract assistant response    if '<|assistant|>' in result:        response = result.split('<|assistant|>')[1]    else:        response = result    print(response[:500])    print()

## Summary### Training Results- Categories: 9- Total reviews loaded: ~90,000 (10,000 per category)- After cleaning: ~65,000 instruction pairs- Model: TinyLlama-1.1B (QLoRA fine-tuned)- GGUF Size: ~668 MB### Next Steps1. Download the GGUF file from Colab2. Upload to S3: `s3://25fltp-ecom-chatbot/model/tinyllama-chat.Q4_K_M.gguf`3. Person 3 will deploy on EC2 via Terraform### Query Types Supported| Type | Description ||------|-------------|| SWOT Analysis | Structured analysis of company strengths, weaknesses, opportunities, threats || Competitor Comparison | Multi-dimensional comparison (Amazon vs Walmart vs Alibaba) || Market Trends | Current and emerging e-commerce trends || Product Category | Analysis of specific product categories || Customer Sentiment | Sentiment analysis of customer reviews || Pricing & Delivery | Analysis of pricing strategies and delivery || Review Intelligence | Deep analysis of review patterns |